In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from configs.config import DATA_DIR, FEWSHOT_CACHE_DIR, PROMPT_DIR, GREEDY_CONFIG
import json
from src.extractor import LabelTransformConfig, prepare_label_tokens, _parse_parent_annotations
    
from src.tokenizer_utils import tokenize, decode
from src.htmlLabel import simplified_to_normal_form
from src.models import get_messages
from tqdm import tqdm


c:\Users\zakga\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Choose the prompting configuration

In [2]:
filename = "2002SCC33"
split = "dev"
filepath = Path(DATA_DIR) / "annotated" / split / f"{filename}.html"
#filepath = Path("output")  / f"{filename}_0.html"

#filepath = PROJECT_ROOT / Path("output") / "dev_gpt5.2_DEC_fs6_greedy_sentence_long" / filename / f"{filename}_2.html"
with open(filepath, "r", encoding="utf-8") as f:
    html_content = f.read()



#### Common Few SHot Selection

In [3]:
fewshot_method = "greedy"   # "greedy" | "random"

fewshot_filename = f"examples_{fewshot_method}_surf-{GREEDY_CONFIG['surface_pattern']}_struct-{GREEDY_CONFIG['structural_pattern']}"
if fewshot_method == "greedy":
    with open(FEWSHOT_CACHE_DIR / f"{fewshot_filename}.json", "r", encoding="utf-8") as f:
        fewshot_file_content = json.load(f)

fewshot_examples = [(example["example"]["input"], example["example"]["output"], example["list_reference_profile"]) for example in fewshot_file_content["examples"]]
print("fewshot examples from :", fewshot_filename)


fewshot examples from : examples_greedy_surf-1.0_struct-0.0


##### Few Shot processing step

In [6]:
from src.reference_profile import ReferenceProfileList
example = fewshot_examples[0]
input, output, list_reference_profile_from_dict = example

rpl = ReferenceProfileList()
rpl = rpl.from_dict(list_reference_profile_from_dict)

output_tokens = tokenize(output)
transformed_output_tokens = prepare_label_tokens(output_tokens, output_label_config)
total_output_text += "|||" + decode(transformed_output_tokens)


In [9]:
filtered_rpl.to_string(attributes=["main_title", "docid", "alternative_titles", "citations", "fragments_mentioned"])

"{'profiles': [{'main_title': 'Renaud v. Quebec (Commission des affaires sociales)', 'docid': 'Renaud', 'alternative_titles': [], 'citations': ['[1999] 3 S.C.R. 855'], 'fragments_mentioned': []}, {'main_title': 'Canada (Minister of Citizenship and Immigration) v. Tobiass', 'docid': 'Tobiass', 'alternative_titles': [], 'citations': ['[1997] 3 S.C.R. 391'], 'fragments_mentioned': ['para.\\xa0108']}, {'main_title': 'ATCO Gas and Pipelines Ltd. v. Alberta (Utilities Commission)', 'docid': 'ATCO Gas and Pipelines', 'alternative_titles': ['ATCO Gas'], 'citations': ['2015 SCC 45', '[2015] 3 S.C.R. 219'], 'fragments_mentioned': []}, {'main_title': 'Barrie Public Utilities v. Canadian Cable Television Assn.', 'docid': 'Barrie Public Utilities', 'alternative_titles': ['Barrie Public Utilities'], 'citations': ['2003 SCC 28', '[2003] 1 S.C.R. 476'], 'fragments_mentioned': ['para. 11', 'paras. 9-19']}, {'main_title': 'Newfoundland and Labrador Nurses’ Union v. Newfoundland and Labrador (Treasury Bo

In [10]:
print(filtered_rpl.to_dict(attributes=["main_title", "docid", "alternative_titles", "citations", "fragments_mentioned"]))

{'profiles': [{'main_title': 'Renaud v. Quebec (Commission des affaires sociales)', 'docid': 'Renaud', 'alternative_titles': [], 'citations': ['[1999] 3 S.C.R. 855'], 'fragments_mentioned': []}, {'main_title': 'Canada (Minister of Citizenship and Immigration) v. Tobiass', 'docid': 'Tobiass', 'alternative_titles': [], 'citations': ['[1997] 3 S.C.R. 391'], 'fragments_mentioned': ['para.\xa0108']}, {'main_title': 'ATCO Gas and Pipelines Ltd. v. Alberta (Utilities Commission)', 'docid': 'ATCO Gas and Pipelines', 'alternative_titles': ['ATCO Gas'], 'citations': ['2015 SCC 45', '[2015] 3 S.C.R. 219'], 'fragments_mentioned': []}, {'main_title': 'Barrie Public Utilities v. Canadian Cable Television Assn.', 'docid': 'Barrie Public Utilities', 'alternative_titles': ['Barrie Public Utilities'], 'citations': ['2003 SCC 28', '[2003] 1 S.C.R. 476'], 'fragments_mentioned': ['para. 11', 'paras. 9-19']}, {'main_title': 'Newfoundland and Labrador Nurses’ Union v. Newfoundland and Labrador (Treasury Boar

In [22]:
import re

from src.htmlLabel import from_simplified

final_fewshot = []
parents_dict = _parse_parent_annotations(total_output_text)
for parent_name, annotations in parents_dict.items():
    for annotation in annotations:
        print(annotation)
        label = from_simplified(simplified_token=tokenize(annotation)[0], label_type='manual_label')
        name = label.name
        docid = label.attributes["docid"]

        filtered_rpl = sample_reference_profile_subset(
            rpl,                 # your ReferenceProfileList
            docid,                # the docid to check for
            doc_type=name,            # optional doc_type filter
            include_docid="random",  # "yes" | "no" | "random"
            length=None,             # OR min_length=2, max_length=8
            min_length=0,
            max_length=5,
            seed=42,
            p=0.8,                # only used when include_docid="random"
        )
        input_annotation = decode(prepare_label_tokens(simplified_to_normal_form(tokenize(annotation),label_type="manual_label"), input_label_config))
        input_rpl = f"Profiles registry : {', '.join(str(profile) for profile in filtered_rpl.profiles)}"
        final_fewshot.append((f"{input_annotation}\n\n{input_rpl}", annotation))

<decision docid="Volvo Canada"><title>Volvo Canada Ltd. v. U.A.W., Local 720</title>, <citation>[1980] 1 S.C.R. 178</citation>, at <fragment>p. 214</fragment></decision>
<decision docid="Toronto"><title>Toronto (City)</title>, at <fragment>paras. 94-95</fragment></decision>
<decision docid="Council of Canadians with Disabilities"><title>VIA Rail</title>, at <fragment>para. 101</fragment></decision>
<decision docid="Mason"><title>Mason v. Minister of Citizenship and Immigration</title>, <citation>2019 FC 1251</citation>, at <fragment>para. 22</fragment></decision>
<decision docid="Irwin Toy"><title>Irwin Toy Ltd. v. Quebec
(Attorney General)</title>, <citation>[1989] 1 S.C.R. 927</citation>, <citation>39 C.R.R. 193</citation></decision>
<decision docid="Généreux"><title>R. v. Généreux</title>, <citation>[1992] 1
S.C.R. 259</citation> at <fragment>310</fragment>, <citation>8 C.R.R. (2d) 89</citation> at <fragment>p. 124</fragment></decision>
<decision docid="Kimble"><title>Kimble</title>

In [16]:
final_fewshot

[("<decision><title>Volvo Canada Ltd. v. U.A.W., Local 720</title>, <citation>[1980] 1 S.C.R. 178</citation>, at <fragment>p. 214</fragment></decision>\n\nProfiles registry : {'doc_type': 'decision', 'jurisdiction': None, 'main_title': 'Volvo Canada Ltd. v. U.A.W., Local 720', 'docid': 'Volvo Canada', 'alternative_titles': [], 'citations': ['[1980] 1 S.C.R. 178'], 'fragments_mentioned': ['p. 214'], 'authors': []}",
  '<decision docid="Volvo Canada"><title>Volvo Canada Ltd. v. U.A.W., Local 720</title>, <citation>[1980] 1 S.C.R. 178</citation>, at <fragment>p. 214</fragment></decision>'),
 ("<decision><title>Toronto (City)</title>, at <fragment>paras. 94-95</fragment></decision>\n\nProfiles registry : {'doc_type': 'decision', 'jurisdiction': None, 'main_title': 'Toronto (City) v. C.U.P.E., Local 79', 'docid': 'Toronto', 'alternative_titles': ['Toronto (City)'], 'citations': ['2003 SCC 63', '[2003] 3 S.C.R. 77'], 'fragments_mentioned': ['para. 62', 'para.\\xa070', 'para. 15', 'para. 131'

In [14]:
import random

from src.reference_profile import ReferenceProfileList


def sample_reference_profile_subset(
    rpl: ReferenceProfileList,
    docid,
    doc_type: str = None,
    include_docid="yes",
    length: int = None,
    min_length: int = None,
    max_length: int = None,
    seed: int = None,
    p: float = 0.7,
) -> ReferenceProfileList:
    """
    Build a random subset of `rpl` as a new ReferenceProfileList.

    Parameters
    ----------
    rpl : ReferenceProfileList
        The full list of profiles to sample from.
    docid :
        The docid we care about when deciding inclusion.
    doc_type : str, optional
        If given, only profiles with this `doc_type` are considered for the subset.
    include_docid : {"yes", "no", "random"}
        - "yes":    the profile with `docid` is forced into the subset.
        - "no":     the profile with `docid` is forced OUT of the subset.
        - "random": the profile with `docid` is included with probability `p`.
    length : int, optional
        Exact size of the returned subset. If given, takes priority over
        min_length/max_length.
    min_length, max_length : int, optional
        If `length` is not given, the subset size is drawn uniformly from
        [min_length, max_length] (inclusive), using `seed`.
    seed : int, optional
        Seed for all the random choices made in this function (size choice,
        whether to include the target docid in "random" mode, and which
        other profiles fill the rest of the subset). Uses a local
        random.Random instance, so global random state is untouched.
    p : float, default 0.7
        Probability of including the target docid's profile when
        include_docid="random". Ignored otherwise.

    Returns
    -------
    ReferenceProfileList
        A new list containing the sampled subset of profiles.

    Raises
    ------
    ValueError
        If include_docid is not one of "yes"/"no"/"random"; if include_docid
        is "yes" but no profile with `docid` exists in `rpl`; if neither
        `length` nor a valid (min_length, max_length) pair is given; or if
        the requested subset size is larger than what's available.
    """
    if include_docid not in ("yes", "no", "random"):
        raise ValueError(
            f"include_docid must be 'yes', 'no', or 'random', got {include_docid!r}"
        )

    rng = random.Random(seed)

    all_profiles = list(rpl)
    if doc_type is not None:
        all_profiles = [prof for prof in all_profiles if prof.doc_type == doc_type]
    target_profile = rpl.get_profile_by_docid(docid)

    if include_docid == "yes" and target_profile is None:
        raise ValueError(f"docid {docid!r} not found in the given ReferenceProfileList")

    # Decide, for this call, whether the target profile should be forced in,
    # forced out, or absent because it doesn't exist.
    force_include_target = False
    force_exclude_target = False

    if target_profile is None:
        # Nothing to force either way; "no" and "random" are trivially satisfied.
        force_exclude_target = True
    elif include_docid == "yes":
        force_include_target = True
    elif include_docid == "no":
        force_exclude_target = True
    else:  # "random"
        if rng.random() < p:
            force_include_target = True
        else:
            force_exclude_target = True

    # Pool of profiles eligible to fill the "free" slots of the subset
    # (everything except the target profile, which is handled separately).
    other_profiles = [prof for prof in all_profiles if prof is not target_profile]

    # Work out the desired subset size.
    # Max possible size of the final subset given the forced inclusion/exclusion:
    if force_include_target:
        max_possible = 1 + len(other_profiles)
    else:
        max_possible = len(other_profiles)

    if length is not None:
        subset_size = length
    else:
        if min_length is None or max_length is None:
            raise ValueError(
                "Either `length`, or both `min_length` and `max_length`, must be provided"
            )
        if min_length > max_length:
            raise ValueError("min_length cannot be greater than max_length")
        subset_size = rng.randint(min_length, max_length)

    if subset_size < 0:
        raise ValueError("Computed subset size is negative")
    if subset_size > max_possible:
        raise ValueError(
            f"Requested subset size ({subset_size}) exceeds the number of "
            f"profiles available under the include_docid={include_docid!r} "
            f"constraint ({max_possible})"
        )

    # How many additional (non-target) profiles do we need to fill the subset?
    remaining_slots = subset_size - 1 if force_include_target else subset_size
    remaining_slots = max(remaining_slots, 0)

    chosen_others = rng.sample(other_profiles, remaining_slots) if remaining_slots > 0 else []

    subset_profiles = list(chosen_others)
    if force_include_target:
        subset_profiles.append(target_profile)

    # Shuffle so the target profile (if forced in) isn't always last.
    rng.shuffle(subset_profiles)

    result = ReferenceProfileList()
    for prof in subset_profiles:
        result.add_profile(prof)

    return result

In [4]:
nb_fewshot_examples = 6
spans_in_context = False

input_label_config = LabelTransformConfig(
    use_simplified=True,
    switch_type=True,
    keep_attributes=["labelname"]
)

output_label_config = LabelTransformConfig(
    use_simplified=True,
    switch_type=True,
    keep_attributes=["labelname", "docid"]
)


# Transform the output in it simplified form
final_fewshot = []
total_output_text = ""
for example in fewshot_examples:
    input, output, list_reference_profile = example

    input_tokens = tokenize(input)
    transformed_input_tokens = prepare_label_tokens(input_tokens, input_label_config)

    output_tokens = tokenize(output)
    transformed_output_tokens = prepare_label_tokens(output_tokens, output_label_config)

    if spans_in_context:
        final_fewshot.append((decode(transformed_input_tokens), decode(transformed_output_tokens)))

    if not spans_in_context:

        total_output_text += "|||" + decode(transformed_output_tokens)

final_fewshot = final_fewshot[:nb_fewshot_examples]


if not spans_in_context:
    parents_dict = _parse_parent_annotations(total_output_text)
    for parent_name, annotations in parents_dict.items():
        for annotation in annotations:
            input = f"{decode(prepare_label_tokens(simplified_to_normal_form(tokenize(annotation),label_type="manual_label"), input_label_config))} Profiles registry : {', '.join(list_reference_profile["profiles"])}"
            if input != annotation:
                final_fewshot.append((input, annotation))

TypeError: sequence item 0: expected str instance, dict found

In [35]:
final_fewshot

[('<legislation><citation>R.S.B.C. 1996, c. 418</citation>, <fragment>s. 159</fragment></legislation>',
  '<legislation docid="Securities Act"><citation>R.S.B.C. 1996, c. 418</citation>, <fragment>s. 159</fragment></legislation>'),
 ('<legislation><fragment>Section 7</fragment> of the <title>Charter</title></legislation>',
  '<legislation docid="Charter"><fragment>Section 7</fragment> of the <title>Charter</title></legislation>'),
 ('<legislation><fragment>s. 7</fragment> of the <title>Charter</title></legislation>',
  '<legislation docid="Charter"><fragment>s. 7</fragment> of the <title>Charter</title></legislation>'),
 ('<legislation><fragment>s. 7</fragment></legislation>',
  '<legislation docid="Charter"><fragment>s. 7</fragment></legislation>'),
 ('<legislation><fragment>s. 7</fragment></legislation>',
  '<legislation docid="Charter"><fragment>s. 7</fragment></legislation>'),
 ('<legislation> <fragment>s. 7</fragment></legislation>',
  '<legislation docid="Charter"> <fragment>s. 7

#### Common Prompt loading

In [29]:
prompt_filename = "coref_long.txt"

with open(PROMPT_DIR / prompt_filename, "r", encoding="utf-8") as f:
    system_prompt = f.read()

print("system_prompt used : ", prompt_filename)

system_prompt used :  coref_long.txt


#### Assistant loading

In [7]:
MODEL_MAPPING_NAME = {
    "qwen7b": "Qwen2.5-7B-Instruct",
    "qwen32b": "Qwen2.5-32B-Instruct",
    "gpt-5.2": "gpt-5.2",
    "phi-4": "phi-4",
    "saul-54b": "SaulLM-54B-Instruct"
}

from src.models import AssistantFactory

model = "gpt-5.2"

if model == "gpt-5.2":
        assistant = AssistantFactory.create_from_config({
            "type": "openai",
            "model_name": model,
            "temperature": 1,
        })
else:
    assistant = AssistantFactory.create(MODEL_MAPPING_NAME[model])


#### Chunk output controle

In [8]:
def process_output(generated, token_chunk, allowed_labels, assistant, with_fallback: bool = True):

    from src.output_control.processor import OutputProcessor
    from src.output_control.fallback import FallbackHandler 
    from src.output_control.verification import VerificationResult 

    controller = OutputProcessor()
    fallback_handler = FallbackHandler(processor=controller)
    
    corrected_generated_tokens, status = controller.process(
        raw_llm_output=generated,
        original_chunk=token_chunk,
        allowed_labels=allowed_labels
    )

    if status.passed:
        return corrected_generated_tokens, status

    if not with_fallback:
        return token_chunk, status
    
    
    corrected_generated_tokens, status_dict = fallback_handler.handle_failure(
        assistant=assistant,
        corrected_output=corrected_generated_tokens,
        original_chunk=token_chunk,
        initial_status=status,
        allowed_labels=allowed_labels,
        fallback_prompt_filename="fallback.txt"
    )
    # Convert dict to VerificationResult
    status = VerificationResult(
        passed=status_dict.get('passed', False),
        error_type=status_dict.get('error_type'),
        details=status_dict.get('error_details'),
        tokens=corrected_generated_tokens
    )
    
    return corrected_generated_tokens, status

### For DEC1-3 

No chunking needed here, we just need the list of mention already labeled

In [32]:
allowed_labels = ["decision", "legislation", "secondary sources", "title", "citation", "source", "authors", "fragment"]

#### Convert into tokens

In [30]:
from src import extract_body, tokenize, clean_tokens
tokens = tokenize(html_content)


#### Get already extracted mention

In [31]:

from src.extractor import build_processing_segments
from src.extractor import get_list_of_mention
from src.models import get_messages
from tqdm import tqdm
from tqdm import tqdm

parent_mentions = get_list_of_mention(
        tokens=tokens,
        keep_labels=["decision", "legislation", "secondary sources"],
        label_type="manual_label"  # Process manual labels from parent extraction
    )

print(f"Found {len(parent_mentions)} parent mentions to process")

segments = build_processing_segments(tokens, parent_mentions)

print(f"Built {len(segments)} token segments "
        f"({sum(s['process'] for s in segments)} to process)")

Found 328 parent mentions to process
Built 657 token segments (328 to process)


#### Main processing function

In [ ]:
config = LabelTransformConfig(
    use_simplified=False,
    switch_type=False,
    keep_attributes=["labelname"]
) # We only remove the attribute
 

failed_count = 0

for idx, segment in enumerate(tqdm(segments, desc="Processing mentions")):
        if not segment["process"]:
            continue


        mention = segment["tokens"]
        html_label = segment["meta"]["label"]

        
        prepared_tokens = prepare_label_tokens(mention, input_label_config)
        user_input = decode(prepared_tokens)

        filtered_fewshot = []
        for example in final_fewshot:
            if example[0].startswith(f"<{html_label.name}>"):
                filtered_fewshot.append(example)
        messages = get_messages(system_prompt=system_prompt, user_input=user_input, fewshot_examples=filtered_fewshot, has_system_role=assistant.has_system_role)


        generated = assistant.generate(messages=messages)
        #print(generated)
        
        corrected_generated_tokens, status = process_output(generated=generated, token_chunk=prepare_label_tokens(mention, config), allowed_labels=allowed_labels, assistant=assistant, with_fallback=False)
        #print(decode(corrected_generated_tokens))
        if not status.passed:
            failed_count += 1

        segment["tokens"] = corrected_generated_tokens

processed_tokens = [
        token
        for segment in segments
        for token in segment["tokens"]
    ]

Processing mentions:   0%|          | 0/557 [00:00<?, ?it/s]

Processing mentions: 100%|██████████| 557/557 [07:46<00:00,  1.19it/s]


#### Post Processing : tokens to HTML

In [43]:
from src.post_processing.main import tokens_to_html_after_decomposed1_3_prompting
processed_html_content = tokens_to_html_after_decomposed1_3_prompting(processed_tokens, html_content)

   ✓ HTMLs match after normalization (ignoring auto_label tags and formatting artifacts)


#### Save File

In [45]:
output_filename = PROJECT_ROOT / Path("output") / "dev_gpt5.2_DEC_fs6_greedy_sentence_long" / filename / f"{filename}_final.html"
#output_filename = Path("output") / f"{filename}_1.html"

with open(output_filename, "w", encoding="utf-8") as f:
    f.write(processed_html_content)